# 02 — Preprocessing & Feature Engineering
**Bank Retention Intelligence Platform**

This notebook covers:
- Loading cleaned data from notebook 01
- Creating 8 engineered features
- Encoding categorical variables
- Saving features dataset to `data/processed/`

> **Input:** `data/processed/cleaned_dataset.csv`  
> **Output:** `data/processed/features_dataset.csv`

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.feature_engineering import engineer_features, get_feature_list
from src.utils import load_processed, save_processed, save_figure, encode_categoricals

print("Libraries loaded")

Libraries loaded


## 1. Load Cleaned Dataset

In [2]:
df = load_processed('cleaned_dataset.csv')
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Loaded: 10,000 rows × 11 columns


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 2. Engineer Features

In [3]:
df = engineer_features(df)

new_features = ['BalanceSalaryRatio', 'ProductsPerTenure', 'EngagementScore',
                'WealthScore', 'RelationshipStrength', 'AgeGroup',
                'WealthSegment', 'HighValueDisengaged']

print("New features created:")
for f in new_features:
    dtype = str(df[f].dtype)
    sample = df[f].iloc[0]
    print(f"  ✓ {f:25s} | dtype: {dtype:10s} | sample: {sample}")

New features created:
  ✓ BalanceSalaryRatio        | dtype: float64    | sample: 0.0
  ✓ ProductsPerTenure         | dtype: float64    | sample: 0.3333333333333333
  ✓ EngagementScore           | dtype: float64    | sample: 1.0
  ✓ WealthScore               | dtype: float64    | sample: 0.9544653075381676
  ✓ RelationshipStrength      | dtype: float64    | sample: 0.46
  ✓ AgeGroup                  | dtype: object     | sample: 36-45
  ✓ WealthSegment             | dtype: object     | sample: Low
  ✓ HighValueDisengaged       | dtype: int32      | sample: 0


## 3. Encode Categorical Variables

In [4]:
df = encode_categoricals(df)
print("Encoding:")
print(f"  Geography → Geography_enc  | unique: {df['Geography_enc'].unique()}")
print(f"  Gender    → Gender_enc     | unique: {df['Gender_enc'].unique()}")
print(f"\nFinal shape: {df.shape}")

Encoding:
  Geography → Geography_enc  | unique: [0 1 2]
  Gender    → Gender_enc     | unique: [0 1]

Final shape: (10000, 21)


## 4. Engineered Feature Statistics

In [5]:
eng_num = ['BalanceSalaryRatio', 'ProductsPerTenure',
           'EngagementScore', 'WealthScore', 'RelationshipStrength']
print("Engineered numerical features — statistics:")
df[eng_num].describe().round(4)

Engineered numerical features — statistics:


,BalanceSalaryRatio,ProductsPerTenure,EngagementScore,WealthScore,RelationshipStrength
count,10000.0000,10000.0000,10000.0000,10000.0000,10000.0000
mean,2.0789,0.3672,0.5722,1.4346,0.4323
std,6.5662,0.3378,0.3741,0.5038,0.1821
min,0.0000,0.0909,0.0000,0.0000,0.0000
25%,0.0000,0.1667,0.3000,1.0766,0.3000
50%,0.7473,0.2500,0.7000,1.4380,0.4300
75%,1.5105,0.5000,1.0000,1.7998,0.5800
max,100.8599,3.0000,1.0000,2.9516,1.0000


## 5. Validate Engagement Score Formula

In [6]:
eng_churn = df.groupby('EngagementScore')['Exited'].mean()*100
print("EngagementScore value → Churn rate:")
for score, churn in eng_churn.sort_index().items():
    label = {0.0: 'Inactive, no card', 0.3: 'Inactive, has card',
             0.7: 'Active, no card', 1.0: 'Active, has card'}.get(round(score,1), str(score))
    print(f"  Score={score:.1f}  ({label:25s}): {churn:.1f}% churn")

EngagementScore value → Churn rate:
  Score=0.0  (Inactive, no card        ): 25.7% churn
  Score=0.3  (Inactive, has card       ): 27.3% churn
  Score=0.7  (Active, no card          ): 16.4% churn
  Score=1.0  (Active, has card         ): 13.4% churn


## 6. Wealth Segment Distribution

In [7]:
ws = df.groupby('WealthSegment').agg(
    Count    =('Exited','count'),
    ChurnRate=('Exited', lambda x: round(x.mean()*100,1)),
    AvgBal   =('Balance', lambda x: round(x.mean(),0))
).reset_index()
print("Wealth Segment breakdown:")
print(ws.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11,4))
order = ['Low','Medium','High','Premium']
ws_ordered = ws.set_index('WealthSegment').reindex(order).reset_index()
colors = ['#3266ad','#1d9e75','#f0a500','#c0392b']
axes[0].bar(ws_ordered['WealthSegment'], ws_ordered['Count'], color=colors, edgecolor='white', width=0.55)
axes[0].set_title('Customers by Wealth Segment', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
bars = axes[1].bar(ws_ordered['WealthSegment'], ws_ordered['ChurnRate'], color=colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, ws_ordered['ChurnRate']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Churn Rate by Wealth Segment', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
plt.tight_layout()
save_figure(fig, 'wealth_segment_churn.png')
plt.show()

Wealth Segment breakdown:
WealthSegment  Count  ChurnRate   AvgBal
         High   2500       22.9  98180.0
          Low   2500       16.4  17278.0
       Medium   2500       18.6  62229.0
      Premium   2500       23.6 127792.0


  Figure saved → E:\bank-retention-intelligence\outputs\figures\wealth_segment_churn.png


## 7. Age Group Distribution

In [8]:
ag = df.groupby('AgeGroup').agg(
    Count    =('Exited','count'),
    ChurnRate=('Exited', lambda x: round(x.mean()*100,1))
).reset_index()
print("Age Group breakdown:")
print(ag.to_string(index=False))

Age Group breakdown:
AgeGroup  Count  ChurnRate
   18-25    611        7.5
   26-35   3542        8.5
   36-45   3736       19.6
   46-55   1311       50.6
     56+    800       36.8


## 8. Relationship Strength vs Churn

In [9]:
df['RSBucket'] = pd.cut(df['RelationshipStrength'], bins=5)
rs_churn = df.groupby('RSBucket', observed=True)['Exited'].mean()*100
print("RelationshipStrength bucket → Churn rate:")
for bucket, rate in rs_churn.items():
    print(f"  {str(bucket):25s}: {rate:.1f}%")

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(range(len(rs_churn)), rs_churn.values, color='#3266ad', edgecolor='white', width=0.65)
ax.set_xticks(range(len(rs_churn)))
ax.set_xticklabels([str(b) for b in rs_churn.index], rotation=25, fontsize=9)
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Relationship Strength vs Churn Rate', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
save_figure(fig, 'relationship_strength_churn.png')
plt.show()
df.drop(columns=['RSBucket'], inplace=True)

RelationshipStrength bucket → Churn rate:
  (-0.001, 0.2]            : 32.1%
  (0.2, 0.4]               : 24.6%
  (0.4, 0.6]               : 16.2%
  (0.6, 0.8]               : 12.3%
  (0.8, 1.0]               : 84.1%


  Figure saved → E:\bank-retention-intelligence\outputs\figures\relationship_strength_churn.png


## 9. Feature Correlation with Churn

In [10]:
feature_cols = get_feature_list()
numeric_df = df[feature_cols + ['Exited']].copy()
corr_with_churn = numeric_df.corr()['Exited'].drop('Exited').sort_values()

print("Correlation with Exited (churn):")
for feat, corr in corr_with_churn.items():
    direction = '↑' if corr > 0 else '↓'
    bar = '█' * int(abs(corr) * 30)
    print(f"  {direction} {feat:25s}: {corr:+.4f}  {bar}")

fig, ax = plt.subplots(figsize=(8,6))
colors_corr = ['#c0392b' if c > 0 else '#3266ad' for c in corr_with_churn.values]
ax.barh(corr_with_churn.index, corr_with_churn.values, color=colors_corr, height=0.6)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with Churn')
ax.set_title('Feature Correlation with Churn', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
save_figure(fig, 'feature_correlation_churn.png')
plt.show()

Correlation with Exited (churn):
  ↓ IsActiveMember           : -0.1561  ████
  ↓ EngagementScore          : -0.1486  ████
  ↓ RelationshipStrength     : -0.1441  ████
  ↓ Gender_enc               : -0.1065  ███
  ↓ NumOfProducts            : -0.0478  █
  ↓ CreditScore              : -0.0245  
  ↓ Tenure                   : -0.0140  
  ↓ HasCrCard                : -0.0071  
  ↓ ProductsPerTenure        : -0.0064  
  ↑ EstimatedSalary          : +0.0121  
  ↑ BalanceSalaryRatio       : +0.0345  █
  ↑ WealthScore              : +0.0739  ██
  ↑ Balance                  : +0.1176  ███
  ↑ Geography_enc            : +0.1538  ████
  ↑ Age                      : +0.2912  ████████


  Figure saved → E:\bank-retention-intelligence\outputs\figures\feature_correlation_churn.png


## 10. Save Features Dataset

In [11]:
save_processed(df, 'features_dataset.csv')
print("\nFeature engineering complete.")
print(f"Total features for modelling: {len(get_feature_list())}")
print("\nFeature list:")
for f in get_feature_list():
    print(f"  • {f}")
print("\nNext: Run 03_clustering.ipynb")

  Saved → E:\bank-retention-intelligence\data\processed\features_dataset.csv  (10,000 rows)

Feature engineering complete.
Total features for modelling: 15

Feature list:
  • CreditScore
  • Age
  • Tenure
  • Balance
  • NumOfProducts
  • HasCrCard
  • IsActiveMember
  • EstimatedSalary
  • BalanceSalaryRatio
  • ProductsPerTenure
  • EngagementScore
  • WealthScore
  • RelationshipStrength
  • Geography_enc
  • Gender_enc

Next: Run 03_clustering.ipynb
